# الفصل السابع عشر: البرمجة غير المتزامنة - مقدمة عملية (async/await)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-17.ipynb)

> **الوقت المتوقع:** ~13 دقيقة قراءة · **13** قسمًا · **17** مثالًا برمجيًا.

**كيف تستخدم هذا الدفتر؟**

1. شغّل الخلايا بالترتيب من الأعلى إلى الأسفل (`Shift` + `Enter`).
2. خلايا الشيفرة قابلة للتعديل: غيّر القيم وأعد التشغيل لتجرّب.
3. خلايا 🧾 معروضة للقراءة فقط (أوامر طرفية، أو إدخال يدوي، أو مقتطف ناقص)، وخلايا ⚠️ تُظهر خطأً مقصودًا يُلتقط تلقائيًا فلا يتوقف «تشغيل الكل».

> **Kaggle:** نزّل الملف من المستودع ثم `File ▸ Import Notebook`، أو استورده مباشرة من GitHub بعد ربط حسابك.

<details>
<summary><b>محتويات الفصل</b></summary>

- **17.1** لماذا يستحق هذا الفصل اهتمامك الكامل؟
- **17.2** فهم المشكلة أولًا: لماذا الكود العادي «بطيء» أحيانًا؟
- **17.3** المفاهيم الأساسية: `async def`, `await`, Coroutines
- **17.4** تحقيق التزامن الحقيقي: `asyncio.gather()`
- **17.5** مثال عملي حقيقي: تحميل بيانات من الإنترنت بـ `aiohttp`
- **17.6** `asyncio.create_task()`: بدء المهام دون انتظار فوري
- **17.7** معالجة الأخطاء في الكود غير المتزامن
- **17.8** متى تستخدم Async، ومتى لا تحتاجه إطلاقًا؟
- **17.9** مشروع تطبيقي: مُجمِّع أسعار من عدة «متاجر» (محاكاة)
- **17.10** تمارين
- **17.11** اختبر فهمك (Quiz)
- **17.12** ملخص الفصل (Cheat Sheet)
- **17.13** ماذا بعد؟

</details>

In [ ]:
#@title ⚙️ تهيئة البيئة  { display-mode: "form" }
import sys, os, random, pathlib

print("Python:", sys.version.split()[0], "|", sys.platform)
assert sys.version_info >= (3, 10), "هذا الكتاب يفترض Python 3.10 أو أحدث"

SEED = 42
random.seed(SEED)

# تثبيت الحزم الناقصة فقط (سريع إن كانت مثبّتة أصلًا)
import importlib.util as _iu
_need = [m for m in ["aiohttp"] if _iu.find_spec(m) is None]
if _need:
    %pip install -q aiohttp

# asyncio.run() لا تعمل داخل Jupyter: نشغّل الحلقة في خيط منفصل
import asyncio as _asyncio, threading as _threading
def _run_in_thread(coro):
    box = {}
    def _target():
        loop = _asyncio.new_event_loop()
        try:
            box['value'] = loop.run_until_complete(coro)
        except BaseException as exc:
            box['error'] = exc
        finally:
            loop.close()
    thread = _threading.Thread(target=_target)
    thread.start()
    thread.join()
    if 'error' in box:
        raise box['error']
    return box['value']
_asyncio.run = _run_in_thread

> ℹ️ أمثلة الشبكة تحتاج إنترنت. عند تعذّره ستفشل خلايا `aiohttp` وحدها.

## 17.1 لماذا يستحق هذا الفصل اهتمامك الكامل؟

تخيّل أنك تعمل في مطعم، وأنت النادل الوحيد. يأتيك زبون ويطلب قهوة. بدلًا من أن تذهب إلى الآلة وتضغط الزر ثم **تقف أمامها تحدّق فيها لمدة 3 دقائق كاملة** حتى تجهز القهوة، ماذا لو ذهبت واستقبلت زبونًا آخر، وأخذت طلبه، وبدأت تحضير طلب ثالث، بينما القهوة الأولى «تُحضَّر في الخلفية»؟ عندما تجهز القهوة، تعود وتقدمها. بهذه الطريقة، تخدم عشرات الزبائن في نفس الوقت الذي كنت ستخدم فيه زبونًا واحدًا فقط لو انتظرت كل طلب بالكامل قبل الانتقال للتالي.

هذا بالضبط هو جوهر **البرمجة غير المتزامنة (Asynchronous Programming)**.

في كل الفصول السابقة، كتبنا كودًا **متزامنًا (Synchronous)**: كل سطر ينتظر السطر الذي قبله حتى ينتهي تمامًا قبل أن يبدأ. هذا منطقي ومثالي عندما تكون العمليات سريعة (جمع رقمين، تكرار قائمة). لكن ماذا يحدث عندما يحتاج برنامجك إلى:

- تحميل بيانات من الإنترنت (قد تستغرق ثوانٍ، وتعتمد على سرعة الشبكة، لا على معالج حاسوبك).
- استدعاء واجهة API خارجية، وانتظار ردها.
- قراءة ملف ضخم من القرص الصلب.
- الانتظار لإدخال من المستخدم دون تجميد باقي البرنامج.

في كل هذه الحالات، معالج حاسوبك **لا يفعل شيئًا فعليًا** أثناء الانتظار — إنه فقط «يقف ساكنًا» ينتظر استجابة من مصدر خارجي (الشبكة، القرص الصلب، إلخ). هذا النوع من الانتظار يُسمى **I/O-bound** (عملية محدودة بالإدخال/الإخراج)، بعكس **CPU-bound** (عملية محدودة بقوة المعالجة الفعلية، كحساب رياضي معقّد).

البرمجة غير المتزامنة تسمح لبرنامجك بـ: «بينما أنتظر هذه العملية البطيئة، دعني أبدأ بعملية أخرى، وسأعود للأولى عندما تجهز». هذا يُحدث فرقًا هائلاً في الأداء عند التعامل مع عشرات أو مئات العمليات المتزامنة (مثل خادم ويب يخدم آلاف الطلبات، أو برنامج يُحمّل 100 صفحة ويب في آن واحد).

> 💡 **نصيحة الخبير:** هذا الفصل هو **مقدمة عملية** وليس تغطية شاملة لكل تعقيدات البرمجة غير المتزامنة (وهي موضوع يمكن أن يملأ كتابًا كاملاً بمفرده!). هدفنا هنا أن تفهم **متى ولماذا** تستخدمها، وأن تكون قادرًا على قراءة وكتابة كود `async/await` أساسي بثقة — وهو ما ستحتاجه بكثرة في الفصل 19 عند بناء واجهات API بـ FastAPI.

## 17.2 فهم المشكلة أولًا: لماذا الكود العادي «بطيء» أحيانًا؟

لنكتب مثالًا يُحاكي عمليات شبكة بطيئة (باستخدام `time.sleep()` كمحاكاة):

In [ ]:
import time

def fetch_data(source: str, delay: float) -> str:
    """يُحاكي طلب بيانات بطيء من مصدر خارجي (مثل API أو قاعدة بيانات)."""
    print(f"🔄 بدء تحميل البيانات من {source}...")
    time.sleep(delay)  # محاكاة وقت الانتظار (شبكة، قرص صلب، إلخ)
    print(f"✅ انتهى تحميل البيانات من {source}")
    return f"بيانات من {source}"

def main() -> None:
    start = time.perf_counter()

    result1 = fetch_data("الخادم أ", 2)
    result2 = fetch_data("الخادم ب", 2)
    result3 = fetch_data("الخادم ج", 2)

    elapsed = time.perf_counter() - start
    print(f"\nالوقت الإجمالي: {elapsed:.2f} ثانية")

main()

الناتج:

```
🔄 بدء تحميل البيانات من الخادم أ...
✅ انتهى تحميل البيانات من الخادم أ
🔄 بدء تحميل البيانات من الخادم ب...
✅ انتهى تحميل البيانات من الخادم ب
🔄 بدء تحميل البيانات من الخادم ج...
✅ انتهى تحميل البيانات من الخادم ج

الوقت الإجمالي: 6.00 ثانية
```

لاحظ: كل عملية تنتظر الأخرى بالكامل، فالوقت الإجمالي هو **مجموع** كل الأوقات (2+2+2=6 ثوانٍ)، رغم أن هذه العمليات **لا تعتمد على بعضها البعض إطلاقًا**! لا يوجد سبب منطقي لانتظار «الخادم أ» قبل البدء بطلب «الخادم ب» — فهما مستقلان تمامًا. هذا هدر واضح للوقت.

> 🔄 **مقارنة: لماذا لا نستخدم Threading أو Multiprocessing هنا؟**
> قد يسألك سؤال وجيه: «ألم نتعلم أن هناك طرقًا أخرى لتنفيذ عمليات متعددة (Threading, Multiprocessing)؟» نعم، لكن:
> - **Multiprocessing** (عمليات متعددة): مناسب للمهام الثقيلة حسابيًا (CPU-bound)، لكنه ثقيل جدًا في استهلاك الذاكرة (كل عملية تحتاج نسخة كاملة من المفسِّر)، وغير مناسب لآلاف العمليات الصغيرة المتزامنة.
> - **Threading** (خيوط): يسمح بالتزامن، لكنه معقّد (مشاكل تزامن الوصول للبيانات المشتركة - Race Conditions)، وفي Python تحديدًا، هناك قيد شهير اسمه **GIL** (Global Interpreter Lock) يمنع تنفيذ أكواد Python الفعلية بالتوازي الحقيقي في نفس اللحظة عبر الخيوط.
> - **Async/Await**: أخف بكثير من Threading (لا حاجة لآلاف الخيوط الثقيلة)، ومصمم خصيصًا لحالات الانتظار الكثيرة (I/O-bound) مثل الشبكة، وهو الخيار الأمثل والأكثر انتشارًا حديثًا لهذا النوع من المشاكل في Python، خاصة في عالم الويب.

## 17.3 المفاهيم الأساسية: `async def`, `await`, Coroutines

### الكلمة المفتاحية `async def`: تعريف دالة غير متزامنة

In [ ]:
async def say_hello() -> None:
    print("مرحبًا!")

بإضافة `async` قبل `def`، تتحول الدالة إلى **Coroutine Function** (دالة مشتركة/تعاونية). عند استدعائها، **لا تُنفَّذ فورًا**:

In [ ]:
result = say_hello()
print(result)  # <coroutine object say_hello at 0x...>
# لاحظ: "مرحبًا!" لم تُطبع إطلاقًا!

استدعاء دالة `async def` **لا يُشغّلها**، بل يُنشئ كائن **Coroutine** — وصفة جاهزة للتنفيذ، لكنها تحتاج «مُشغِّلًا» خاصًا ليبدأ تنفيذها فعليًا.

### تشغيل أول Coroutine باستخدام `asyncio.run()`

In [ ]:
import asyncio

async def say_hello() -> None:
    print("مرحبًا!")

asyncio.run(say_hello())  # الآن تُطبع "مرحبًا!" فعليًا

`asyncio` هي وحدة مدمجة في Python (لا تحتاج تثبيتًا) توفّر كل الأدوات اللازمة للبرمجة غير المتزامنة. `asyncio.run()` هي **نقطة الدخول** الرئيسية: تُنشئ «حلقة أحداث» (Event Loop — سنشرحها قريبًا)، تُشغّل الـ Coroutine المُعطاة، ثم تُغلق كل شيء عند الانتهاء.

> 💡 **نصيحة الخبير:** `asyncio.run()` يجب أن تُستدعى **مرة واحدة فقط** في برنامجك، عادة في نقطة الدخول الرئيسية (`if __name__ == "__main__":`). لا تستدعِها من داخل Coroutine أخرى.

### الكلمة المفتاحية `await`: انتظار نتيجة Coroutine أخرى

In [ ]:
import asyncio

async def greet(name: str) -> None:
    print(f"مرحبًا، {name}!")

async def main() -> None:
    await greet("أحمد")
    await greet("سارة")

asyncio.run(main())

`await` تعني: «نفّذ هذه الـ Coroutine، وانتظر نتيجتها، لكن أثناء الانتظار، **اسمح لعمليات أخرى بالعمل** إن كانت تنتظر أيضًا». هذا الجزء الأخير هو **جوهر** الفائدة الكاملة من `async/await`.

> ⚠️ **قاعدة صارمة:** لا يمكنك استخدام `await` إلا داخل دالة معرّفة بـ `async def`. محاولة استخدامها خارج ذلك تُصدر `SyntaxError`.
> ```python
> def regular_function():
>     await something()  # ❌ SyntaxError: 'await' outside async function
> ```

### محاكاة الانتظار غير المتزامن: `asyncio.sleep()`

تذكر أننا استخدمنا `time.sleep()` في المثال الأول لمحاكاة بطء الشبكة. لكن `time.sleep()` هي دالة **متزامنة (Blocking)** — تُجمّد **كل البرنامج بالكامل**، ولا تسمح لأي كود آخر بالعمل أثناء انتظارها، حتى لو كان الكود الآخر Coroutine!

الحل: `asyncio.sleep()` — نسخة غير متزامنة من `sleep()`، مصممة خصيصًا للعمل مع `async/await`:

In [ ]:
import asyncio
import time

async def fetch_data(source: str, delay: float) -> str:
    print(f"🔄 بدء تحميل البيانات من {source}...")
    await asyncio.sleep(delay)  # ← غير متزامن! يسمح لعمليات أخرى بالعمل أثناء الانتظار
    print(f"✅ انتهى تحميل البيانات من {source}")
    return f"بيانات من {source}"

async def main() -> None:
    start = time.perf_counter()

    result1 = await fetch_data("الخادم أ", 2)
    result2 = await fetch_data("الخادم ب", 2)
    result3 = await fetch_data("الخادم ج", 2)

    elapsed = time.perf_counter() - start
    print(f"\nالوقت الإجمالي: {elapsed:.2f} ثانية")

asyncio.run(main())

لكن انتظر... النتيجة هنا **لا تزال 6 ثوانٍ تقريبًا**! لماذا؟

> ⚠️ **سوء فهم شائع جدًا للمبتدئين:** مجرد استخدام `async`/`await` **لا يجعل العمليات تعمل بالتوازي تلقائيًا**! في المثال أعلاه، كتبنا `await fetch_data(...)` ثلاث مرات **متتالية** — هذا يعني «انتظر الأولى حتى تنتهي كاملاً، **ثم** ابدأ الثانية». لا يوجد تزامن حقيقي هنا بعد! لتحقيق التزامن الفعلي، نحتاج أداة إضافية: `asyncio.gather()`.

## 17.4 تحقيق التزامن الحقيقي: `asyncio.gather()`

In [ ]:
import asyncio
import time

async def fetch_data(source: str, delay: float) -> str:
    print(f"🔄 بدء تحميل البيانات من {source}...")
    await asyncio.sleep(delay)
    print(f"✅ انتهى تحميل البيانات من {source}")
    return f"بيانات من {source}"

async def main() -> None:
    start = time.perf_counter()

    results = await asyncio.gather(
        fetch_data("الخادم أ", 2),
        fetch_data("الخادم ب", 2),
        fetch_data("الخادم ج", 2),
    )

    elapsed = time.perf_counter() - start
    print(f"\nالنتائج: {results}")
    print(f"الوقت الإجمالي: {elapsed:.2f} ثانية")

asyncio.run(main())

الناتج:

```
🔄 بدء تحميل البيانات من الخادم أ...
🔄 بدء تحميل البيانات من الخادم ب...
🔄 بدء تحميل البيانات من الخادم ج...
✅ انتهى تحميل البيانات من الخادم أ
✅ انتهى تحميل البيانات من الخادم ب
✅ انتهى تحميل البيانات من الخادم ج

النتائج: ['بيانات من الخادم أ', 'بيانات من الخادم ب', 'بيانات من الخادم ج']
الوقت الإجمالي: 2.00 ثانية
```

هذا هو الفرق المذهل! الآن الثلاث عمليات **بدأت في نفس الوقت تقريبًا**، وانتظرنا جميعها معًا، فكان الوقت الإجمالي **2 ثانية فقط** (بدلًا من 6)! هذا لأن `asyncio.gather()` تُشغّل كل الـ Coroutines الممرَّرة إليها بشكل متزامن (Concurrent)، وتنتظرها جميعًا، ثم تُرجع كل النتائج في قائمة واحدة، **بنفس ترتيب** الـ Coroutines الأصلية.

> 💡 **نصيحة الخبير:** `asyncio.gather()` هي الأداة الأكثر استخدامًا عمليًا في `asyncio`. القاعدة الذهبية: **لا تستخدم `await` بشكل متتالٍ لعمليات مستقلة عن بعضها البعض** — استخدم `gather()` لتشغيلها معًا. استخدم `await` المتتالي فقط عندما تحتاج فعليًا نتيجة العملية الأولى **قبل** أن تبدأ الثانية (اعتمادية حقيقية بين العمليات).

### فهم ما يحدث «خلف الكواليس»: حلقة الأحداث (Event Loop)

لفهم السحر الذي يحدث، تخيّل **حلقة الأحداث (Event Loop)** كمدير مهام ذكي جدًا:

1. يبدأ تنفيذ `fetch_data("الخادم أ", 2)`، يصل إلى `await asyncio.sleep(2)`.
2. بدلًا من التجميد والانتظار، يقول Event Loop: «حسنًا، هذه المهمة ستنتظر ثانيتين، دعني أضعها جانبًا وأتفقد مهام أخرى جاهزة للعمل الآن».
3. يبدأ `fetch_data("الخادم ب", 2)`، ثم `fetch_data("الخادم ج", 2)`، بنفس المنطق.
4. بعد ثانيتين تقريبًا، تكون كل المهام الثلاث قد «انتهت من انتظارها»، فيعود Event Loop لإكمال كل واحدة بالترتيب الذي أصبحت فيه جاهزة.

هذا كله يحدث في **خيط تنفيذ واحد فقط (Single Thread)**! لا يوجد تعدد معالجات حقيقي هنا، بل **تبديل ذكي** بين المهام المنتظرة، يستغل وقت الانتظار (الذي كان سيُهدَر) لتنفيذ مهام أخرى.

> 🔄 **مقارنة: التزامن (Concurrency) مقابل التوازي (Parallelism)**
> - **Concurrency (تزامن)**: عدة مهام «تتقدم معًا» خلال فترة زمنية، لكن ليس بالضرورة في نفس اللحظة الدقيقة (قد يكون التبديل بينها سريعًا جدًا فيبدو الأمر متزامنًا). هذا ما يفعله `asyncio`.
> - **Parallelism (توازي)**: عدة مهام تُنفَّذ **فعليًا في نفس اللحظة** على أنوية معالج مختلفة (هذا ما يوفره `multiprocessing`).
> - `asyncio` مثالي لمهام I/O-bound الكثيرة (شبكة، ملفات) لأن وقت الانتظار يُستغل بذكاء. لكنه **لا يُسرّع** مهام CPU-bound (حسابات ثقيلة)، لأن هذه المهام تحتاج فعلاً وقت معالج حقيقي، لا مجرد «انتظار».

## 17.5 مثال عملي حقيقي: تحميل بيانات من الإنترنت بـ `aiohttp`

حتى الآن استخدمنا `asyncio.sleep()` كمحاكاة. لنرَ مثالًا حقيقيًا: تحميل محتوى عدة صفحات ويب في آن واحد، باستخدام مكتبة `aiohttp` (نسخة غير متزامنة من مكتبات طلبات HTTP، بعكس `requests` التي سنستخدمها في الفصل 21 وهي متزامنة بطبيعتها).

### التثبيت

```bash
uv add aiohttp
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### المثال

In [ ]:
import asyncio
import time
import aiohttp

async def fetch_url(session: aiohttp.ClientSession, url: str) -> int:
    """يُحمّل صفحة ويب ويُرجع رمز الحالة (Status Code) فقط، للتبسيط."""
    async with session.get(url) as response:
        return response.status

async def main() -> None:
    urls = [
        "https://www.python.org",
        "https://www.github.com",
        "https://www.wikipedia.org",
    ]

    start = time.perf_counter()

    async with aiohttp.ClientSession() as session:
        tasks = [fetch_url(session, url) for url in urls]
        results = await asyncio.gather(*tasks)

    elapsed = time.perf_counter() - start

    for url, status in zip(urls, results):
        print(f"{url} → {status}")

    print(f"\nالوقت الإجمالي: {elapsed:.2f} ثانية")

asyncio.run(main())

لاحظ النقاط التالية:
- `async with` تُستخدم مع `aiohttp.ClientSession()` — نسخة غير متزامنة من `with` التي تعلمناها في الفصل 16، تعمل مع Context Managers غير متزامنة (`__aenter__`/`__aexit__`).
- `tasks = [fetch_url(session, url) for url in urls]`: قائمة من Coroutines (لم تُشغَّل بعد).
- `*tasks`: نستخدم عامل فك التغليف (من الفصل 10) لتمرير كل عناصر القائمة كمعاملات منفصلة لـ `gather()`.

تشغيل هذا البرنامج سيُحمّل الصفحات الثلاث **في نفس الوقت تقريبًا**، بدلًا من تحميلها واحدة تلو الأخرى، مما يُقلل الوقت الإجمالي بشكل ملحوظ (خاصة مع عدد أكبر من الروابط).

> 💡 **نصيحة الخبير:** كل مكتبة I/O «عادية» (متزامنة) لها عادة «نسخة توأم» غير متزامنة: `requests` ↔ `aiohttp` أو `httpx` (التي تدعم الاثنين معًا)، `psycopg2` ↔ `asyncpg` لقواعد بيانات PostgreSQL، إلخ. عند العمل في بيئة `async` (مثل تطبيق FastAPI)، **يجب** استخدام النسخ غير المتزامنة من كل مكتبة، وإلا ستُجمّد استدعاءاتها المتزامنة حلقة الأحداث بالكامل وتُلغي كل فائدة `asyncio`!

> ⚠️ **خطأ فادح وشائع جدًا:** استخدام دالة متزامنة بطيئة (مثل `time.sleep()` أو `requests.get()`) داخل Coroutine! هذا يُجمّد **حلقة الأحداث بأكملها**، فتتوقف كل المهام الأخرى المنتظرة أيضًا، مما يُلغي كل فائدة `async` تمامًا:
> ```python
> import time
>
> async def bad_example() -> None:
>     time.sleep(2)  # ❌ خطأ فادح! يُجمّد حلقة الأحداث بأكملها
>
> async def good_example() -> None:
>     await asyncio.sleep(2)  # ✅ صحيح، يسمح بتبديل المهام
> ```

## 17.6 `asyncio.create_task()`: بدء المهام دون انتظار فوري

أحيانًا تريد أن «تبدأ» عملية غير متزامنة، لكن تريد الاستمرار في عمل آخر، والعودة لنتيجتها لاحقًا. هنا تأتي `asyncio.create_task()`:

In [ ]:
import asyncio

async def background_job(name: str, delay: float) -> str:
    print(f"🚀 بدأت المهمة: {name}")
    await asyncio.sleep(delay)
    print(f"🏁 انتهت المهمة: {name}")
    return f"نتيجة {name}"

async def main() -> None:
    # إنشاء المهمة تبدأ تنفيذها فورًا "في الخلفية"
    task1 = asyncio.create_task(background_job("أ", 3))
    task2 = asyncio.create_task(background_job("ب", 1))

    print("البرنامج يستمر بعمل أشياء أخرى بينما المهام تعمل في الخلفية...")
    await asyncio.sleep(0.5)
    print("لا يزال البرنامج يعمل...")

    # الآن ننتظر نتائج المهام عند الحاجة الفعلية إليها
    result1 = await task1
    result2 = await task2

    print(result1, result2)

asyncio.run(main())

الفرق بين `create_task()` و`gather()`: `create_task()` يُنشئ المهمة وتبدأ **فورًا** في الخلفية، تاركًا إياك حرًا لتنفيذ كود آخر قبل أن تنتظر نتيجتها بـ `await` لاحقًا. بينما `gather()` تُشغّل كل الـ Coroutines **وتنتظرها معًا** في نفس اللحظة دون القيام بأي شيء آخر بينهما.

## 17.7 معالجة الأخطاء في الكود غير المتزامن

الأخبار الجيدة: `try/except` تعمل بنفس الطريقة تمامًا مع الكود غير المتزامن!

In [ ]:
import asyncio

async def risky_operation(should_fail: bool) -> str:
    await asyncio.sleep(1)
    if should_fail:
        raise ValueError("حدث خطأ أثناء العملية!")
    return "نجحت العملية"

async def main() -> None:
    try:
        result = await risky_operation(should_fail=True)
        print(result)
    except ValueError as e:
        print(f"تم اصطياد الخطأ: {e}")

asyncio.run(main())

### التعامل مع الأخطاء عند استخدام `gather()`

In [ ]:
import asyncio

async def task_that_fails() -> str:
    await asyncio.sleep(1)
    raise ValueError("فشلت هذه المهمة!")

async def task_that_succeeds() -> str:
    await asyncio.sleep(1)
    return "نجحت هذه المهمة"

async def main() -> None:
    results = await asyncio.gather(
        task_that_succeeds(),
        task_that_fails(),
        return_exceptions=True,  # ← مهم جدًا!
    )

    for result in results:
        if isinstance(result, Exception):
            print(f"خطأ: {result}")
        else:
            print(f"نجاح: {result}")

asyncio.run(main())

> ⚠️ **تحذير مهم:** بدون `return_exceptions=True`، فإن `asyncio.gather()` تتوقف فورًا وتُطلق الاستثناء الأول الذي يحدث في **أي** مهمة من المهام، حتى لو كانت المهام الأخرى لا تزال تعمل (قد تستمر في الخلفية دون أن تحصل على نتائجها). باستخدام `return_exceptions=True`، تحصل على **كل** النتائج، بما فيها كائنات الاستثناءات نفسها، لتتعامل معها بنفسك واحدة تلو الأخرى.

## 17.8 متى تستخدم Async، ومتى لا تحتاجه إطلاقًا؟

### 🔄 جدول القرار: Sync أم Async؟

| الموقف | الخيار المناسب | السبب |
|---|---|---|
| حسابات رياضية ثقيلة (معالجة صور، تدريب نماذج AI) | Sync (أو `multiprocessing`) | CPU-bound، لا يوجد «انتظار» مفيد لاستغلاله |
| استدعاء عشرات/مئات طلبات API | **Async** | I/O-bound، الانتظار هو عنق الزجاجة |
| قراءة ملف واحد صغير | Sync | لا فائدة من التعقيد الإضافي لعملية واحدة بسيطة |
| خادم ويب يخدم آلاف الطلبات المتزامنة | **Async** (مثل FastAPI) | يسمح بخدمة طلبات كثيرة بخيط واحد بكفاءة عالية |
| سكريبت بسيط يُشغَّل مرة واحدة ويقرأ ملفًا | Sync | البساطة أهم هنا من كل تحسين أداء غير ملموس |
| Web Scraping لمئات الصفحات (الفصل 21) | **Async** | تسريع هائل بسبب كثرة طلبات الشبكة المستقلة |

> ⚠️ **تحذير من «فخ التعقيد غير المبرر»:** لا تستخدم `async/await` في كل مكان «لأنها حديثة وعصرية»! إن كان برنامجك يقوم بعملية واحدة فقط، أو عمليات متسلسلة تعتمد كل منها على سابقتها، فإن Async **تضيف تعقيدًا بلا أي فائدة حقيقية**. اسأل نفسك دائمًا: «هل لدي عدة عمليات I/O **مستقلة** يمكن أن تعمل معًا؟» إن كانت الإجابة لا، التزم بالكود المتزامن البسيط الذي أتقنته طوال هذا الكتاب.

> 💡 **نصيحة الخبير:** لا يمكنك ولا يجب عليك «تحويل» كل كودك إلى Async. عمليًا، تجد غالبًا **نقطة دخول واحدة غير متزامنة** (مثل خادم الويب في FastAPI) تستدعي من داخلها كودًا متزامنًا عاديًا للعمليات التي لا تحتاج async (كالحسابات البسيطة)، وتستخدم `await` فقط عند التعامل مع عمليات I/O فعلية (قاعدة بيانات، طلبات خارجية).

## 17.9 مشروع تطبيقي: مُجمِّع أسعار من عدة «متاجر» (محاكاة)

لنبني مشروعًا يُحاكي سيناريو واقعيًا جدًا: تطبيق يقارن أسعار منتج واحد من عدة متاجر إلكترونية مختلفة، عبر استدعاءات شبكة (محاكاة)، مستفيدًا من التزامن لتسريع العملية بشكل كبير.

### المتطلبات

1. محاكاة استدعاء "API" لكل متجر (بأوقات استجابة مختلفة، بل وبعضها قد «يفشل»).
2. تشغيل كل الاستدعاءات بشكل متزامن باستخدام `asyncio.gather()`.
3. معالجة أخطاء المتاجر الفاشلة دون إيقاف باقي العملية.
4. عرض تقرير نهائي: أرخص سعر، أغلى سعر، والمتوسط.
5. قياس ومقارنة الوقت المستغرق بين النسخة المتزامنة (Sync) وغير المتزامنة (Async).

<details>
<summary>✅ الحل النموذجي الكامل</summary>

```python
"""
مُجمِّع أسعار - مقارنة أداء Sync مقابل Async
"""
from __future__ import annotations

import asyncio
import random
import time
from dataclasses import dataclass

@dataclass
class StoreResult:
    store_name: str
    price: float | None  # None يعني فشل الاستعلام
    error: str | None = None

# ---------------------------------------------------------
# محاكاة استدعاء API لمتجر واحد (غير متزامن)
# ---------------------------------------------------------
async def fetch_price_from_store(store_name: str, base_delay: float) -> StoreResult:
    """يُحاكي استعلام سعر من متجر معيّن، مع احتمال فشل عشوائي."""
    # محاكاة تذبذب في زمن الاستجابة (كالشبكة الحقيقية)
    actual_delay = base_delay + random.uniform(-0.3, 0.3)
    await asyncio.sleep(max(actual_delay, 0.1))

    # محاكاة فشل عشوائي بنسبة 20%
    if random.random() < 0.2:
        return StoreResult(store_name, price=None, error="انتهت مهلة الاتصال")

    price = round(random.uniform(80, 150), 2)
    return StoreResult(store_name, price=price)

# ---------------------------------------------------------
# النسخة غير المتزامنة: كل المتاجر معًا
# ---------------------------------------------------------
async def compare_prices_async(stores: dict[str, float]) -> list[StoreResult]:
    tasks = [
        fetch_price_from_store(name, delay)
        for name, delay in stores.items()
    ]
    return await asyncio.gather(*tasks)

# ---------------------------------------------------------
# النسخة المتزامنة التقليدية (للمقارنة فقط)
# ---------------------------------------------------------
def fetch_price_sync(store_name: str, base_delay: float) -> StoreResult:
    actual_delay = base_delay + random.uniform(-0.3, 0.3)
    time.sleep(max(actual_delay, 0.1))

    if random.random() < 0.2:
        return StoreResult(store_name, price=None, error="انتهت مهلة الاتصال")

    price = round(random.uniform(80, 150), 2)
    return StoreResult(store_name, price=price)

def compare_prices_sync(stores: dict[str, float]) -> list[StoreResult]:
    return [fetch_price_sync(name, delay) for name, delay in stores.items()]

# ---------------------------------------------------------
# تحليل وعرض النتائج
# ---------------------------------------------------------
def print_report(results: list[StoreResult]) -> None:
    print("\n--- تقرير الأسعار ---")
    successful: list[StoreResult] = []

    for result in results:
        if result.price is not None:
            print(f"  ✅ {result.store_name}: {result.price} دولار")
            successful.append(result)
        else:
            print(f"  ❌ {result.store_name}: فشل ({result.error})")

    if not successful:
        print("لم ينجح أي استعلام!")
        return

    prices = [r.price for r in successful if r.price is not None]
    cheapest = min(successful, key=lambda r: r.price or float("inf"))
    most_expensive = max(successful, key=lambda r: r.price or 0)

    print(f"\n💰 الأرخص: {cheapest.store_name} ({cheapest.price} دولار)")
    print(f"💸 الأغلى: {most_expensive.store_name} ({most_expensive.price} دولار)")
    print(f"📊 المتوسط: {sum(prices) / len(prices):.2f} دولار")

# ---------------------------------------------------------
# البرنامج الرئيسي: مقارنة الأداء
# ---------------------------------------------------------
async def main() -> None:
    stores: dict[str, float] = {
        "متجر الأمانة": 1.5,
        "سوق النجاح": 2.0,
        "بازار الخير": 1.0,
        "متجر السرعة": 0.8,
        "سوق التوفير": 1.8,
    }

    print("=" * 50)
    print("🐢 تشغيل النسخة المتزامنة (Sync)...")
    start_sync = time.perf_counter()
    sync_results = compare_prices_sync(stores)
    sync_elapsed = time.perf_counter() - start_sync
    print_report(sync_results)
    print(f"\n⏱️  وقت النسخة المتزامنة: {sync_elapsed:.2f} ثانية")

    print("\n" + "=" * 50)
    print("🚀 تشغيل النسخة غير المتزامنة (Async)...")
    start_async = time.perf_counter()
    async_results = await compare_prices_async(stores)
    async_elapsed = time.perf_counter() - start_async
    print_report(async_results)
    print(f"\n⏱️  وقت النسخة غير المتزامنة: {async_elapsed:.2f} ثانية")

    print("\n" + "=" * 50)
    speedup = sync_elapsed / async_elapsed if async_elapsed > 0 else 0
    print(f"🎯 التسريع المحقق: {speedup:.1f}x أسرع!")

if __name__ == "__main__":
    asyncio.run(main())
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

نتيجة متوقعة (الأرقام الدقيقة تختلف بسبب العشوائية، لكن الفارق الكبير في الوقت ثابت):

```
⏱️  وقت النسخة المتزامنة: 7.21 ثانية
⏱️  وقت النسخة غير المتزامنة: 2.05 ثانية
🎯 التسريع المحقق: 3.5x أسرع!
```

> 💡 لاحظ كيف أن النسخة المتزامنة استغرقت تقريبًا **مجموع** كل أوقات الانتظار، بينما النسخة غير المتزامنة استغرقت تقريبًا **وقت أطول تأخير فردي فقط** — هذا هو بالضبط الفرق الذي يجعل Async خيارًا حاسمًا في تطبيقات الويب الحديثة وأي نظام يتعامل مع عمليات شبكة متعددة.

</details>

## 17.10 تمارين

### 🟢 مستوى مبتدئ

**تمرين 1:** اكتب Coroutine باسم `countdown(n: int) -> None` تطبع عدًا تنازليًا من `n` إلى 1، مع `await asyncio.sleep(1)` بين كل رقم والذي يليه. شغّلها باستخدام `asyncio.run()`.

**تمرين 2:** ما الفرق بين `await func()` و `asyncio.create_task(func())`؟ اشرح بجملة واحدة لكل منهما.

**تمرين 3:** صحّح هذا الكود الذي يحتوي خطأ شائعًا:

In [ ]:
async def get_data() -> str:
    return "بيانات"

def main():
    result = get_data()
    print(result)

main()

<details>
<summary>✅ الحل النموذجي</summary>

In [ ]:
# تمرين 1
import asyncio

async def countdown(n: int) -> None:
    for i in range(n, 0, -1):
        print(i)
        await asyncio.sleep(1)
    print("انتهى!")

asyncio.run(countdown(5))

# تمرين 2
# await func(): ينفّذ الـ Coroutine وينتظر نتيجتها فورًا قبل المتابعة.
# asyncio.create_task(func()): يبدأ تنفيذ الـ Coroutine "في الخلفية" فورًا،
# ويسمح لك بمتابعة كود آخر، على أن تنتظر نتيجتها لاحقًا بـ await عند الحاجة.

# تمرين 3 - الكود المصحح
import asyncio

async def get_data() -> str:
    return "بيانات"

async def main() -> None:
    result = await get_data()  # يجب await لتنفيذ الـ Coroutine فعليًا
    print(result)

asyncio.run(main())  # يجب استدعاء main عبر asyncio.run، وليس مباشرة

</details>


### 🟡 مستوى متوسط

**تمرين 4:** اكتب 3 Coroutines تُحاكي تحميل: صورة (3 ثوانٍ)، فيديو (5 ثوانٍ)، ومستند (1 ثانية). شغّلها معًا بـ `asyncio.gather()`، واطبع الوقت الإجمالي. تأكد أن الوقت الإجمالي قريب من 5 ثوانٍ فقط (أطول عملية)، وليس 9 ثوانٍ (المجموع).

**تمرين 5:** عدّل المثال السابق بحيث تفشل عملية «تحميل الفيديو» (ترفع `ConnectionError`). استخدم `return_exceptions=True` في `gather()`، واطبع رسالة مناسبة لكل عملية، سواء نجحت أم فشلت.

**تمرين 6:** اكتب دالة `fetch_with_timeout(delay: float, timeout: float) -> str` تستخدم `asyncio.wait_for()` لفرض مهلة قصوى على عملية `asyncio.sleep(delay)`. إن تجاوز `delay` قيمة `timeout`، يجب اصطياد `asyncio.TimeoutError` وإرجاع رسالة «انتهت المهلة».

<details>
<summary>✅ الحل النموذجي</summary>

In [ ]:
# تمرين 4
import asyncio
import time

async def download_image() -> str:
    await asyncio.sleep(3)
    return "تم تحميل الصورة"

async def download_video() -> str:
    await asyncio.sleep(5)
    return "تم تحميل الفيديو"

async def download_document() -> str:
    await asyncio.sleep(1)
    return "تم تحميل المستند"

async def main() -> None:
    start = time.perf_counter()
    results = await asyncio.gather(
        download_image(), download_video(), download_document()
    )
    elapsed = time.perf_counter() - start
    print(results)
    print(f"الوقت: {elapsed:.2f} ثانية")  # قريب من 5 ثوانٍ

asyncio.run(main())

# تمرين 5
import asyncio

async def download_image() -> str:
    await asyncio.sleep(3)
    return "تم تحميل الصورة"

async def download_video() -> str:
    await asyncio.sleep(2)
    raise ConnectionError("فشل الاتصال بخادم الفيديو")

async def download_document() -> str:
    await asyncio.sleep(1)
    return "تم تحميل المستند"

async def main() -> None:
    results = await asyncio.gather(
        download_image(), download_video(), download_document(),
        return_exceptions=True,
    )
    for result in results:
        if isinstance(result, Exception):
            print(f"❌ فشل: {result}")
        else:
            print(f"✅ نجح: {result}")

asyncio.run(main())

# تمرين 6
import asyncio

async def fetch_with_timeout(delay: float, timeout: float) -> str:
    try:
        async def task() -> str:
            await asyncio.sleep(delay)
            return "تم التحميل بنجاح"
        return await asyncio.wait_for(task(), timeout=timeout)
    except asyncio.TimeoutError:
        return "انتهت المهلة"

async def main() -> None:
    print(await fetch_with_timeout(1, 3))   # "تم التحميل بنجاح"
    print(await fetch_with_timeout(5, 2))   # "انتهت المهلة"

asyncio.run(main())

</details>


### 🔴 تحدي

**تمرين 7:** ابنِ «نظام فحص صحة خوادم (Health Check)»:
- دالة `check_server(name: str, is_healthy: bool, delay: float) -> dict[str, object]` تُحاكي فحص خادم، تُرجع قاموسًا `{"name": ..., "status": "healthy"/"down", "response_time": ...}`.
- دالة `check_all_servers(servers: list[tuple[str, bool, float]]) -> list[dict[str, object]]` تفحص كل الخوادم معًا (Async)، وتُرجع النتائج.
- دالة تحليلية تطبع: عدد الخوادم السليمة، عدد المعطلة، وأبطأ خادم استجابةً من بين السليمة.
- استخدم `asyncio.wait_for()` لفرض مهلة قصوى 3 ثوانٍ على كل فحص؛ إن تجاوزها الخادم، يُعتبر "down" بسبب "timeout".

<details>
<summary>✅ الحل النموذجي</summary>

In [ ]:
import asyncio
from typing import TypedDict

class ServerStatus(TypedDict):
    name: str
    status: str
    response_time: float | None

async def check_server(name: str, is_healthy: bool, delay: float) -> ServerStatus:
    async def do_check() -> float:
        await asyncio.sleep(delay)
        if not is_healthy:
            raise ConnectionError("الخادم لا يستجيب")
        return delay

    try:
        response_time = await asyncio.wait_for(do_check(), timeout=3.0)
        return {"name": name, "status": "healthy", "response_time": response_time}
    except asyncio.TimeoutError:
        return {"name": name, "status": "down", "response_time": None}
    except ConnectionError:
        return {"name": name, "status": "down", "response_time": None}

async def check_all_servers(
    servers: list[tuple[str, bool, float]]
) -> list[ServerStatus]:
    tasks = [check_server(name, healthy, delay) for name, healthy, delay in servers]
    return await asyncio.gather(*tasks)

def analyze_results(results: list[ServerStatus]) -> None:
    healthy = [r for r in results if r["status"] == "healthy"]
    down = [r for r in results if r["status"] == "down"]

    print(f"✅ الخوادم السليمة: {len(healthy)}")
    for server in healthy:
        print(f"   {server['name']} - {server['response_time']:.2f}s")

    print(f"❌ الخوادم المعطلة: {len(down)}")
    for server in down:
        print(f"   {server['name']}")

    if healthy:
        slowest = max(healthy, key=lambda r: r["response_time"] or 0)
        print(f"\n🐢 أبطأ خادم سليم: {slowest['name']} ({slowest['response_time']:.2f}s)")

async def main() -> None:
    servers = [
        ("خادم الويب الرئيسي", True, 0.5),
        ("خادم قاعدة البيانات", True, 1.2),
        ("خادم الكاش", False, 1.0),
        ("خادم البريد", True, 4.0),  # سيتجاوز المهلة (3 ثوانٍ) → down
    ]

    results = await check_all_servers(servers)
    analyze_results(results)

if __name__ == "__main__":
    asyncio.run(main())

</details>

## 17.11 اختبر فهمك (Quiz)

<details>
<summary>❓ السؤال 1: ما الفرق بين I/O-bound و CPU-bound؟ ولماذا يهم هذا الفرق عند اختيار Async؟</summary>

I/O-bound: العملية تقضي معظم وقتها في الانتظار (شبكة، قرص صلب) وليس في معالجة فعلية. CPU-bound: العملية تستهلك فعليًا قدرة المعالج (حسابات معقّدة). Async مفيد جدًا للحالة الأولى لأنه يستغل وقت الانتظار لتنفيذ مهام أخرى، لكنه لا يُسرّع الحالة الثانية لأن المعالج يعمل فعليًا طوال الوقت دون «انتظار» يمكن استغلاله.
</details>

<details>
<summary>❓ السؤال 2: ماذا يحدث عند استدعاء دالة `async def` دون استخدام `await` أو `asyncio.run()`؟</summary>

لا يُنفَّذ جسم الدالة إطلاقًا! الاستدعاء يُنشئ فقط كائن Coroutine (وصفة جاهزة للتنفيذ)، لكنه لا يُشغّل الكود الفعلي حتى يُمرَّر لـ `await` داخل Coroutine أخرى، أو إلى `asyncio.run()` كنقطة دخول رئيسية.
</details>

<details>
<summary>❓ السؤال 3: لماذا يُعتبر استخدام `time.sleep()` داخل دالة `async def` خطأً فادحًا؟</summary>

لأن `time.sleep()` دالة متزامنة (Blocking) تُجمّد **حلقة الأحداث بأكملها**، مانعة أي Coroutine أخرى من العمل أثناء الانتظار، مما يُلغي كل فائدة استخدام async أصلًا. يجب استخدام `asyncio.sleep()` بدلًا منها، لأنها تسمح لحلقة الأحداث بالتبديل إلى مهام أخرى جاهزة أثناء الانتظار.
</details>

<details>
<summary>❓ السؤال 4: ما الفرق بين تنفيذ عدة `await` متتالية وتنفيذها عبر `asyncio.gather()`؟</summary>

تنفيذ `await` متتالية (كل واحدة في سطر منفصل) يعني انتظار كل عملية بالكامل قبل بدء التي تليها — لا تزامن حقيقي، والوقت الإجمالي هو مجموع كل الأوقات. بينما `asyncio.gather()` تُشغّل كل الـ Coroutines الممرَّرة إليها في نفس الوقت تقريبًا، فيكون الوقت الإجمالي قريبًا من أطول عملية فردية فقط، وليس مجموعها.
</details>

<details>
<summary>❓ السؤال 5: متى يجب أن تتجنّب استخدام async/await في مشروعك؟</summary>

عندما لا توجد عمليات I/O متعددة ومستقلة يمكن تنفيذها بالتزامن (مثل سكريبت بسيط يقوم بعملية واحدة، أو عمليات متسلسلة يعتمد بعضها على نتيجة سابقتها بشكل كامل)، أو عند التعامل مع مهام CPU-bound ثقيلة حسابيًا، حيث لا توجد فائدة حقيقية من async وتضيف تعقيدًا غير مبرر.
</details>

## 17.12 ملخص الفصل (Cheat Sheet)

```python
import asyncio

# ===== تعريف وتشغيل Coroutine =====
async def my_function() -> str:
    await asyncio.sleep(1)   # انتظار غير متزامن (وليس time.sleep!)
    return "النتيجة"

asyncio.run(my_function())   # نقطة الدخول الرئيسية - تُستدعى مرة واحدة فقط

# ===== انتظار متتالٍ (عند الاعتماد الحقيقي بين العمليات) =====
async def main() -> None:
    result1 = await my_function()
    result2 = await my_function()  # يبدأ فقط بعد انتهاء الأولى

# ===== تزامن حقيقي (عمليات مستقلة) =====
async def main() -> None:
    results = await asyncio.gather(
        my_function(),
        my_function(),
        return_exceptions=True,  # لمنع توقف الكل عند فشل واحدة
    )

# ===== بدء مهمة في الخلفية =====
async def main() -> None:
    task = asyncio.create_task(my_function())
    # ... كود آخر يعمل الآن ...
    result = await task  # ننتظر النتيجة عند الحاجة فقط

# ===== فرض مهلة قصوى =====
try:
    result = await asyncio.wait_for(my_function(), timeout=2.0)
except asyncio.TimeoutError:
    print("انتهت المهلة")

# ===== معالجة الأخطاء =====
try:
    result = await risky_coroutine()
except SomeError as e:
    print(f"خطأ: {e}")
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `SomeError`، `risky_coroutine` من مثال سابق.

| المفهوم | الاستخدام |
|---|---|
| `async def` | تعريف Coroutine Function |
| `await` | تنفيذ وانتظار Coroutine أخرى (داخل async def فقط) |
| `asyncio.run()` | نقطة الدخول - تُستدعى مرة واحدة |
| `asyncio.sleep()` | انتظار غير متزامن (بديل `time.sleep`) |
| `asyncio.gather()` | تشغيل عدة Coroutines بالتزامن وانتظار الكل |
| `asyncio.create_task()` | بدء مهمة في الخلفية فورًا |
| `asyncio.wait_for()` | فرض مهلة قصوى على عملية |
| `return_exceptions=True` | منع توقف `gather()` عند أول خطأ |

## 17.13 ماذا بعد؟

لقد فتحت للتو بابًا أساسيًا في عالم Python الحديث: فهم متى ولماذا نحتاج التزامن، وكيف نكتبه بأمان عبر `async/await`. هذه المهارة ستكون **جوهرية** عندما نبني أول واجهة API حقيقية في الفصل 19 باستخدام FastAPI، حيث يعتمد كل شيء تقريبًا على هذا النمط.

لكن قبل أن ننتقل لبناء مشاريع حقيقية أكبر، هناك مهارة عملية أساسية لا بد من إتقانها أولًا: كيف نُنظّم مشاريعنا بشكل احترافي؟ كيف نُدير التبعيات (المكتبات الخارجية) بطريقة نظيفة وقابلة للمشاركة مع الآخرين؟ وكيف نستخدم أداة التحكم بالإصدارات **Git** — الأداة التي يعتمد عليها كل مطور محترف في العالم لحفظ تاريخ تطور كوده والتعاون مع فريقه؟

في الفصل الثامن عشر: **إدارة المشاريع كمحترف**، سنتعمق في Virtual Environments، أداة `uv` بكل إمكانياتها (التي استخدمناها بشكل أساسي طوال الكتاب)، هيكلة مشروع Python احترافية قياسية، وأساسيات Git الضرورية لكل مبرمج.

---

### الخطوة التالية

- [الفصل الثامن عشر: إدارة المشاريع كمحترف](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-18.ipynb)
- [فهرس كل الدفاتر](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/00-index.ipynb)

*الفصل السابع عشر — من كتاب «بايثون 3.14: من الصفر إلى الاحتراف».*